In [28]:
import pandas as pd
import numpy as np
import re
import os

from sklearn.model_selection import train_test_split
SEED = 42

In [29]:
import re
import emoji

slang_dict = {
    "ga": "tidak",
    "gak": "tidak",
    "gk": "tidak",
    "nggak": "tidak",
    "tp": "tapi",
    "jd": "jadi",
    "bgt": "banget",
    "yg": "yang",
    "krn": "karena"
}

def normalize_slang(text):
    words = text.split()
    return " ".join([slang_dict.get(w, w) for w in words])


def preprocess_text(text):
    text = str(text)

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URL
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # 3. Remove mention
    text = re.sub(r'@\w+', ' ', text)

    # 4. Remove hashtag symbol only (kata tetap)
    text = re.sub(r'#', '', text)

    # 5. Remove emoji
    text = emoji.replace_emoji(text, replace='')

    # 6. Slang normalization ringan
    text = normalize_slang(text)

    # 7. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [ ]:
df_sent = pd.read_csv("comment_wlabel_sentiment.csv", encoding="latin1")

df_sent["comment"] = df_sent["comment"].astype(str).apply(preprocess_text)
df_sent["label"] = pd.to_numeric(df_sent["label"], errors="coerce")

df_sent = df_sent.dropna(subset=["label"])
df_sent = df_sent[df_sent["label"].isin([0,1,2])].reset_index(drop=True)

train_df, test_df = train_test_split(
    df_sent,
    test_size=0.3,
    stratify=df_sent["label"],
    random_state=SEED
)


In [31]:
df_sent["comment"] = df_sent["comment"].astype(str)
df_sent["comment"] = df_sent["comment"].fillna("")
df_sent["label"] = df_sent["label"].astype(int)

In [ ]:
print(df_sent["comment"].head())
print(type(df_sent["comment"].iloc[0]))

0    temen-temen yang mau prompt yang kupakai silah...
1                                      harus pake vpn?
2    mas,bahas tune ai mas,peraturannya agak ribet,...
3    email asli saya pernah menggunakan trial tapi ...
4                        pikiran nya udah jaaht duluan
Name: comment, dtype: str
<class 'str'>


In [39]:
os.makedirs("data", exist_ok=True)

train_df.to_csv("data/train.csv", index=False)
test_df.to_csv("data/test.csv", index=False)